In [1]:
# --- 1. LIBRARIES ---
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_curve, auc)

# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (Dense, Conv1D, BatchNormalization, 
                                     concatenate, Flatten, Dropout, Attention, 
                                     Reshape, GlobalAveragePooling1D, Multiply, MaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# --- 2. REPRODUCIBILITY (The "Same Result" Rule) ---
def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)
    print(f"Environment seeds locked at {seed}")

set_seeds(42)

Environment seeds locked at 42


In [3]:
# Load the specific binary dataset
df = pd.read_csv("leave_BoT_IoT_DDoS.csv")

# Verify the load by checking the total rows and columns
print(f"Dataset successfully loaded.")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

Dataset successfully loaded.
Total Rows: 65091331
Total Columns: 84


In [4]:
df.columns

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts',
       'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
       'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std',
       'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean',
       'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean',
       'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot',
       'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
       'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',
       'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags',
       'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Packets/s',
       'Bwd Packets/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean',
       'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt',
       'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt',
       'CWE Flag Count

In [5]:
# 1. Strip any leading/trailing whitespace and convert to lowercase
df.columns = df.columns.str.strip().str.lower()

# 2. Replace spaces, slashes, and dots with underscores for easy coding
df.columns = df.columns.str.replace(' ', '_', regex=False)
df.columns = df.columns.str.replace('/', '_', regex=False)
df.columns = df.columns.str.replace('.', '_', regex=False)

# 3. Print the first 10 columns to verify the change
print("First 10 standardized columns:")
print(df.columns[:10].tolist())

# 4. Find the exact name of your Label column
label_col = [col for col in df.columns if 'label' in col]
print(f"\nLabel column found: {label_col}")

First 10 standardized columns:
['flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts']

Label column found: ['label']


In [6]:
# List of columns that are not useful for training features
identifiers = ['flow_id', 'src_ip', 'dst_ip', 'timestamp']

# Drop them only if they exist in the current dataframe
df.drop(columns=[col for col in identifiers if col in df.columns], inplace=True)

print(f"Identifiers removed.")
print(f"Remaining columns: {df.shape[1]}")

Identifiers removed.
Remaining columns: 80


In [7]:
rows_with_nan = df.isna().any(axis=1).sum()

print(f"Total rows with at least one NaN: {rows_with_nan}")

Total rows with at least one NaN: 1403540


In [8]:
# ==========================================================
from sklearn.linear_model import LinearRegression

print("Starting Regression Imputation for Infinity values...")

# 1. Replace all Inf with NaN as per the paper's methodology
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 2. Identify columns with and without NaN
all_features = df.drop(columns=['label']).columns
COLS_WNA = df.columns[df.isna().any()].tolist()
COLS_WONA = [c for c in all_features if c not in COLS_WNA]

# 3. Regression Imputation Loop
for target_col in COLS_WNA:
    # Train on rows where the target is not null
    train_data = df[df[target_col].notna()]
    predict_data = df[df[target_col].isna()]
    
    if len(predict_data) > 0:
        model = LinearRegression()
        # Train model using 'Clean' columns to predict the 'Broken' column
        model.fit(train_data[COLS_WONA], train_data[target_col])
        
        # Fill missing values with predicted values
        predictions = model.predict(predict_data[COLS_WONA])
        df.loc[df[target_col].isna(), target_col] = predictions
        
print(f"Imputation Complete. Total NaN remaining: {df.isna().sum().sum()}")
# ==========================================================

Starting Regression Imputation for Infinity values...
Imputation Complete. Total NaN remaining: 0


In [9]:
import joblib

# Dictionary to store the regression model for each column
imputer_models = {}

for target_col in COLS_WNA:
    train_data = df[df[target_col].notna()]
    if not train_data.empty:
        model = LinearRegression()
        model.fit(train_data[COLS_WONA], train_data[target_col])
        
        # SAVE the model into our dictionary
        imputer_models[target_col] = model
        
        # Fill training NaNs
        predict_data = df[df[target_col].isna()]
        if not predict_data.empty:
            predictions = model.predict(predict_data[COLS_WONA])
            df.loc[df[target_col].isna(), target_col] = predictions

# Save the entire dictionary of models to a file
joblib.dump({
    'models': imputer_models,
    'cols_wona': COLS_WONA,
    'cols_wna': COLS_WNA
}, "regression_imputer.pkl")

print("Regression Imputer models saved to regression_imputer.pkl")

Regression Imputer models saved to regression_imputer.pkl


In [10]:
# 1. Check for any columns that are not integers or floats (excluding 'label')
non_numeric = df.drop(columns=['label']).select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric columns found (excluding label): {non_numeric}")

# 2. Check for Infinite values (common in this dataset)
inf_count = np.isinf(df.select_dtypes(include=[np.number])).values.sum()
print(f"Total Infinite values: {inf_count}")

# 3. Check for Null/NaN values
nan_count = df.isnull().sum().sum()
print(f"Total NaN values: {nan_count}")

Non-numeric columns found (excluding label): []
Total Infinite values: 0
Total NaN values: 0


In [11]:
# 1. Check unique values in the label column
print("Unique labels in dataset:")
print(df['label'].unique())

# 2. Check the count of each label to see if it is imbalanced
print("\nLabel counts:")
print(df['label'].value_counts())

# 3. Check the data type of the label
print(f"\nLabel data type: {df['label'].dtype}")

Unique labels in dataset:
[1 0]

Label counts:
label
1    55388814
0     9702517
Name: count, dtype: int64

Label data type: int64


In [12]:
# 1. Define Features (X) and Target (y)
X = df.drop(columns=['label'])
y = df['label']

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Verification of the split
print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set:  {X_test.shape[0]} rows")

Training set: 52073064 rows
Testing set:  13018267 rows


In [13]:
# 1. Initialize the Scaler
scaler = MinMaxScaler()

# 2. Fit on training data and transform both sets
# This preserves the default float64 precision
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Verification of the results
print(f"Scaling Complete.")
print(f"Data Type:         {X_train_scaled.dtype}")
print(f"Number of Features: {X_train_scaled.shape[1]}")
print(f"Value Range:       {X_train_scaled.min()} to {X_train_scaled.max()}")

Scaling Complete.
Data Type:         float64
Number of Features: 79
Value Range:       0.0 to 1.0


In [14]:
# --- STAGE 1: AUTOENCODER DEFINITION ---

# 1. Use your 79 features as the input dimension
input_dim = X_train_scaled.shape[1]
ae_in = Input(shape=(input_dim,))

# 2. Encoder: Bottleneck strategy (96 neurons -> 32 neurons)
enc = Dense(96, activation='relu')(ae_in)
enc = BatchNormalization()(enc)
enc = Dense(32, activation='relu')(enc) 

# 3. AE Attention Mechanism
# Reshaping to (1, 32) allows the Attention layer to weigh the encoded features
att_reshape = Reshape((1, 32))(enc)
att_logic = Attention()([att_reshape, att_reshape])
att_flat = Flatten()(att_logic)

# 4. Decoder: Reconstruction strategy
# Maps the 32 features back to the original 79 dimensions
dec = Dense(96, activation='tanh')(att_flat)
dec_out = Dense(input_dim, activation='sigmoid')(dec) 

# 5. Model Creation
# 'autoencoder' is for training; 'encoder_only' is for feature extraction
autoencoder = Model(ae_in, dec_out, name="Autoencoder")
encoder_only = Model(ae_in, att_flat, name="Encoder_Extractor")

# 6. Compilation
# Using Adam optimizer and Mean Absolute Error (MAE) loss
autoencoder.compile(optimizer='adam', loss='mae')

# Display the architecture
autoencoder.summary()

Model: "Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 79)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 96)                │           7,680 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 96)                │             384 │ dense[0][0]                │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 32)                │           3,104 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ reshape (Reshape)             │ (None, 1, 32)             │               0 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention (Attention)         │ (None, 1, 32)             │               0 │ reshape[0][0],             │
│                               │                           │                 │ reshape[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 32)                │               0 │ attention[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 96)                │           3,168 │ flatten[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_3 (Dense)               │ (None, 79)                │           7,663 │ dense_2[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 21,999 (85.93 KB)

 Trainable params: 21,807 (85.18 KB)

 Non-trainable params: 192 (768.00 B)

In [15]:
# 1. Define Early Stopping to monitor validation loss
early_stop_ae = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

# 2. Train the model
# Input (X_train_scaled) is also the Target (X_train_scaled) for an Autoencoder
print("Starting Autoencoder Training...")
history_ae = autoencoder.fit(
    X_train_scaled, X_train_scaled,
    epochs=30,
    batch_size=1024,
    validation_split=0.2,
    callbacks=[early_stop_ae],
    verbose=1
)

print("\nAutoencoder training finished.")

Starting Autoencoder Training...
Epoch 1/30


C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


40683/40683 ━━━━━━━━━━━━━━━━━━━━ 434s 5ms/step - loss: 0.0039 - val_loss: 0.0027
Epoch 2/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 199s 5ms/step - loss: 0.0025 - val_loss: 0.0024
Epoch 3/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 199s 5ms/step - loss: 0.0024 - val_loss: 0.0023
Epoch 4/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 196s 5ms/step - loss: 0.0024 - val_loss: 0.0023
Epoch 5/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 196s 5ms/step - loss: 0.0023 - val_loss: 0.0023
Epoch 6/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 200s 5ms/step - loss: 0.0023 - val_loss: 0.0022
Epoch 7/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 197s 5ms/step - loss: 0.0023 - val_loss: 0.0022
Epoch 8/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 193s 5ms/step - loss: 0.0023 - val_loss: 0.0022
Epoch 9/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 194s 5ms/step - loss: 0.0022 - val_loss: 0.0022
Epoch 10/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 194s 5ms/step - loss: 0.0022 - val_loss: 0.0022
Epoch 11/30
40683/40683 ━━━━━━━━━━━━━━━━━━━━ 197s 5ms/step - loss: 0.0022 - val_loss: 0.00

In [17]:
import joblib

# 1. Save the Scaler (essential for normalizing new data correctly)
joblib.dump(scaler, "scaler.pkl")

# 2. Save the Encoder (the 32-feature extractor)
encoder_only.save("AE_Encoder_Extractor.keras")

# 3. Save the full Autoencoder (in case you want to fine-tune it later)
autoencoder.save("AE_Stage1_Full.keras")

print("All components saved successfully:")
print("- scaler.pkl (Feature Normalization)")
print("- AE_Encoder_Extractor.keras (Feature Extraction)")
print("- AE_Stage1_Full.keras (Full Autoencoder)")

All components saved successfully:
- scaler.pkl (Feature Normalization)
- AE_Encoder_Extractor.keras (Feature Extraction)
- AE_Stage1_Full.keras (Full Autoencoder)
